<a href="https://colab.research.google.com/github/simulate111/Computer-vision2026ABO/blob/main/CompViz_part_I_nr_3_ipynb_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IT00CJ11 Computer Vision (5 cr. ECTS), spring 2025

***Direct computational methods and deep learning for vision***

***Another look at the Singular Value Decomposition, (SVD)***

The Singular Value Decomposition (SVD) is a linear algebra technique where a matrix is factored into product of three matrices, that is

$$\mathbf{A} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^{\mathrm{T}} \ ,$$

where $\mathbf{\Sigma}$ is a diagonal matrix whose entries are called singular values. Interestingly for an image, only the top few singular values contain most of the "information" to represent the image. For further information cf: https://en.wikipedia.org/wiki/Singular_value_decomposition

One application of SVD is data compression. Given a data matrix $\mathbf{A}$ (for instance an image), SVD can help to find a low rank matrix which is a good approximation of the original data matrix.

Below we have code for a very basic image compression algorithm. Given a colored image, the algorithm will calculate the singular value decomposition of the image matrix. Then it will find the optimal number of dimensions required to get best tradeoff between reconstruction error and image fidelity. As a quality measure, we use the Frobenius Norm to calculate the reconstruction error.

In this post, we will see step-by-step example of performing SVD on an image and use top singular vectors or principal components to reconstruct it.

Let us load the packages needed to perform SVD on images. In addition to the standard NumPy, we need PIL for image manipulation.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

We work with our familiar image, roadmap.jpg. Start with loading it

In [2]:
img = Image.open("roadmap.jpg")

FileNotFoundError: [Errno 2] No such file or directory: 'roadmap.jpg'

Let us check the image by opening the image object using show() function from Image module. This will open the image in a separate window.

In [3]:
img.show()
img.layers

NameError: name 'img' is not defined

We can also check the image size with size(). It will give you the width and height of the image.

In [4]:
img.size

NameError: name 'img' is not defined

An image is stored as a matrix. An image in RGB color model stores an image in three matrices one each for Red, Green, and Blue color. In this post on using SVD to reconstruct an image, we will only deal with one of the color matrices, not all three. However, the same principle holds for dealing with other two matrices as well.

Let us first extract image corresponding to Red color matrix. With PIL we can get a color matrix using getdata() function. We specify band=0 to get red color image.

In [ ]:
aimg = np.array(img)
aimg[:,:,1] *=0
aimg[:,:,2] *=0
a = Image.fromarray(aimg)
a.show()
red_band = Image.Image.getdata(a,band=0)

Let us convert the red color image as a numpy array containing the numerical values corresponding to each pixel i.e. matrix element.

In [ ]:
# convert to numpy array
img_mat = np.array(list(red_band), float)
img_mat.size
img_mat.shape

Now we will convert the one dimensional numpy array and 2d-numpy matrix using the image size.

In [ ]:
# get image shape
img_mat.shape = (img.size[1], img.size[0])
# convert 1d-array to matrix
img_mat = np.matrix(img_mat)
img_mat

Let us check how the numpy matrix looks like as image using PIL’s imshow() function.

In [ ]:
plt.imshow(a)

Let us also compare how the original RGB image looks in comparison to the image using the single layer of the image by plotting side-by-side.

In [ ]:
fig, axs = plt.subplots(1, 2,figsize=(10,10))
axs[0].imshow(img)
axs[0].set_title('Original Image', size=16)
axs[1].imshow(a)
axs[1].set_title(' "R" band image', size=16)
plt.tight_layout()
plt.savefig('Original_image_and_R_band_image_for_SVD.jpg',dpi=150)

Let us center and scale the data before applying SVD. This will help us put each variable in the same scale.

In [ ]:
# scale the image matrix befor SVD
img_mat_scaled= (img_mat-img_mat.mean())/img_mat.std()

We can use NumPy’s linalg module’s svd function to perform singular value decomposition (SVD) on the scaled image matrix.

In [ ]:
# Perform SVD using np.linalg.svd
U, s, V = np.linalg.svd(img_mat_scaled)

Performing singular value decomposition (SVD) on matrix will factorize or decompose the matrix in three matrices, $\mathbf{U}$, $\mathbf{\Sigma}$, and $\mathbf{V}$. The columns of both $\mathbf{U}$ and $\mathbf{V}$ matrices are orthonormal and called right and left singular vectors. And the matrix $\mathbf{\Sigma}$ is a diagonal matrix with only positive numbers and corresponding to the eigenvalues of $\mathbf{A}$. Note also the dimensions:
$$\begin{align*} \dim{ \mathbf{A} } &= m \times n \ , \\
\dim{ \mathbf{U} } &= m \times p \ , \ \ \text{orthogonal} \\
\dim{ \mathbf{\Sigma} } &= p \times p \ , \ \ \text{diagonal} \\
\dim{ \mathbf{V} } &= n \times p \ , \ \ \text{orthogonal} \ .
\end{align*}$$

Let us check the dimension of U and V matrices. We can see that both U and V are square matrices and their dimensions matches the image size.

In [ ]:
U.shape

In [5]:
V.shape

NameError: name 'V' is not defined

We can also see that the routine svd produces only the diagonal elements of $\mathbf{\Sigma}$, which are the eigenvalues of $\mathbf{A}$ ordered in descending magnitude as a vector.

In [ ]:
s

We can use these eigenvalues from SVD to compute the amount of variances explained by each singular vector contained in $\mathbf{U}$ and $\mathbf{V}$

In [ ]:
# Compute Variance explained by each singular vector
var_explained = np.round(s**2/np.sum(s**2), decimals=3)

The first singular vector or principal component explains most of variation in the image. In this example it explains 45% of the total variation and the second one explains close to 11% of the variation.

In [ ]:
# Variance explained by top singular vectors
var_explained[0:20]

A bar plot made using variance explained nicely captures how each vector is contributing to the variation in the image.

In [ ]:
x=list(range(1,21))
y=var_explained[0:20]
plt.bar(x, y)
plt.xlabel('Singular Vector', fontsize=16)
plt.ylabel('Variance Explained', fontsize=16)
plt.tight_layout()
plt.savefig('svd_scree_plot.png',dpi=150)

*Reconstructing Image with top-K Singular vectors*

The top K singular vectors captures most of the variation. Therefore instead of using all the singular vectors and multiplying them as shown in SVD decomposition, we can reconstruct the image using the K largest (in magnitude) singular vectors.

Let us use the top 5 singular vectors and reconstruct the matrix using matrix multiplication in the decomposition shown above. Let us also visualize the reconstructed image.

In [ ]:
num_components = 5
reconst_img_5 = np.matrix(U[:, :num_components]) * np.diag(s[:num_components]) * np.matrix(V[:num_components, :])
plt.imshow(reconst_img_5)
plt.savefig('reconstructed_image_with_5_SVs.png',dpi=150)

We can see that the top 5 components is not enough to reconstruct the image, let us use top 50 singular vectors and see how the reconstructed image look like.

In [6]:
num_components = 50
reconst_img_50 = np.matrix(U[:, :num_components]) * np.diag(s[:num_components]) * np.matrix(V[:num_components, :])
plt.imshow(reconst_img_50)
plt.title('Reconstructed Image: 50 SVs', size=16)
plt.savefig('reconstructed_image_with_50_SVs.png',dpi=150)

NameError: name 'U' is not defined

The quality of the reconstructed image improves as we use more top singular vectors. Here is a comparison of the reconstructed image using different numbers of components.

In [7]:
reconst_img_5 = np.matrix(U[:, :5]) * np.diag(s[:5]) * np.matrix(V[:5, :])
reconst_img_50 = np.matrix(U[:, :50]) * np.diag(s[:50]) * np.matrix(V[:50, :])
reconst_img_100 = np.matrix(U[:, :100]) * np.diag(s[:100]) * np.matrix(V[:100, :])
reconst_img_500 = np.matrix(U[:, :500]) * np.diag(s[:500]) * np.matrix(V[:500, :])
fig, axs = plt.subplots(2, 2)
axs[0, 0].imshow(reconst_img_5)
axs[0, 0].set_title('Reconstructed Image: 5 SVs', size=8)
axs[0, 1].imshow(reconst_img_50)
axs[0, 1].set_title('Reconstructed Image: 50 SVs', size=8)
axs[1, 0].imshow(reconst_img_100)
axs[1, 0].set_title('Reconstructed Image: 100 SVs', size=8)
axs[1, 1].imshow(reconst_img_500)
axs[1, 1].set_title('Reconstructed Image: 500 SVs', size=8)
plt.tight_layout()
plt.savefig('reconstructed_images_using_different_SVs.jpg',dpi=350)

NameError: name 'U' is not defined

We can see the improvement in the quality of image as we add more singular vectors initally and then it saturates suggesting that we don’t gain much adding more components as the variance explained becomes small.

**Using the Pickle module**

If you want to save some results or data for later use, the pickle module, which comes with Python, is very useful. Pickle can take almost any Python object and convert it to a string representation. This process is called *pickling*. Reconstructing the object from the string representation is conversely called *unpickling*. This string representation can
then be easily stored or transmitted. Let’s illustrate this with an example. Suppose we want to save the image eigenvalues and svd matrix of the previous section. This is done like this:

In [ ]:
import pickle
# save eigenvalues and svd matrix
f = open('font_pca_modes.pkl', 'wb')
pickle.dump(s,f)
pickle.dump(U,f)
f.close()

As you can see, several objects can be pickled to the same file. There are several different protocols available for the .pkl files, and if unsure, it is best to read andwrite binary files. To load the data in some other Python session, just use the load() method like this:

In [ ]:
# load eigenvalues and svd matrix
f = open('font_pca_modes.pkl', 'rb')
s = pickle.load(f)
U = pickle.load(f)
f.close()

Note that the order of the objects should be the same! There is also an optimized version written in C called cpickle that is fully compatible with the standard pickle module.

For the remainder of this book, we will use the with statement to handle file reading and writing. This is a construct that was introduced in Python 2.5 that automatically handles opening and closing of files (even if errors occur while the files are open). Here is what the saving and loading above looks like using with():

In [ ]:
# open file and save
with open('font_pca_modes.pkl', 'wb') as f:
  pickle.dump(s,f)
  pickle.dump(U,f)

and:

In [ ]:
# open file and load
with open('font_pca_modes.pkl', 'rb') as f:
  s = pickle.load(f)
  U = pickle.load(f)

This might look strange the first time you see it, but it is a very useful construct. If you don’t like it, just use the open and close functions as above. As an alternative to using pickle, NumPy also has simple functions for reading andwriting
text files that can be useful if your data does not contain complicated structures, for example a list of points clicked in an image. To save an array x to file, use:

In [ ]:
np.savetxt('test.txt',s,'%i')

The last parameter indicates that integer format should be used. Similarly, reading is done like this:

In [ ]:
s = np.loadtxt('test.txt')

Finally, NumPy has dedicated functions for saving and loading arrays. Look for save() and load() in the documentation for the details.

**SciPy**

SciPy (http://scipy.org/) is an open-source package for mathematics that builds on NumPy and provides efficient routines for a number of operations, including numerical integration, optimization, statistics, signal processing, and most importantly for us, image processing. As the following will show, there are many useful modules in SciPy. SciPy is free and available at http://scipy.org/Download.

*Blurring images*

A classic and very useful example of image convolution is Gaussian blurring of images. In essence, the (grayscale) image $I$ is convolved with a Gaussian kernel $G_\sigma$ to create a blurred version:
$$I_\sigma = I * G_\sigma = \iint_{(x,y)} I(x-s,y-t) G_\sigma(s,t) \ \mathrm{d}s \mathrm{d}t$$
where the *Gaussian kernel* with standard deviation $\sigma$ is defined as:
$$G_\sigma(s,t) = \frac{1}{2 \pi \sigma} \mathrm{e}^{-\frac{s^2+t^2}{2 \sigma^2}}$$
Gaussian blurring is used to define an image scale to work in, for interpolation, for computing interest points, and in many more applications. SciPy comes with a module for filtering called scipy.ndimage.filters that can be used to compute these convolutions using a fast 1D separation. All you need to do is this:

In [ ]:
import sys
sys.executable
!C:\\Users\\tfredman\\opencv\\Scripts\\python.exe -m pip install scipy

In [ ]:
import scipy
from PIL import Image
from numpy import *
from scipy.ndimage import gaussian_filter
im = array(Image.open('roadmap.jpg').convert('L'))
im2 = gaussian_filter(im,10)
plt.imshow(im2,cmap='gray')

Here the last parameter of gaussian_filter() is the standard deviation. Try filters with increasing $\sigma$. Larger values give less detail. To blur color images, simply apply Gaussian blurring to each color channel:

In [ ]:
im = array(Image.open('roadmap.jpg'))
im2 = zeros(im.shape)
for i in range(3):
  im2[:,:,i] = gaussian_filter(im[:,:,i],5)
im2 = uint8(im2)
plt.imshow(im2)

**Image derivatives**

How the image intensity changes over the image is important information and is used for many applications, as we will see throughout this book. The intensity change is described with the $x$ and $y$ derivatives $I_x$ and $I_y$ of the graylevel image $I$ (for color images, derivatives are usually taken for each color channel).
The image gradient is the vector $\nabla I = \left[ I_x , \ I_y \right]^{\mathrm{T}}$. The gradient has two important
properties, the gradient magnitude
$$\left| \nabla I \right| = \sqrt{I_x^2 + I_y^2}\ ,$$
describing the strength of the intensity change, and the *gradient angle*:
$$\alpha = \arctan{\left( \frac{I_y}{I_x} \right)}$$
indicating the direction of the largest intensity change at each pixel in the image. The NumPy function arctan2() returns the signed angle in radians from the interval $[-\pi , \ \pi]$.

Computing the image derivatives can be done using discrete approximations. These are implemented as convolutions
$$I_x = I * D_x \ , \ \ I_y = I * D_y \ .$$
These convolutions are obtained as follows. Consider the simple situation:
![image.png](attachment:image.png)
On the left is the image matrix: each pixel is marked with its value. The initial pixel has a red border. The kernel action area has a green border. In the middle is the kernel and, on the right is the convolution result.

Here is what happened: the filter read successively, from left to right and from top to bottom, all the pixels of the kernel action area. It multiplied the value of each of them by the kernel corresponding value and added results. The initial pixel has become $42$: $(40 \cdot 0)+(42 \cdot 1)+(46 \cdot 0) + (46 \cdot 0)+(50 \cdot 0)+(55 \cdot 0) + (52 \cdot 0)+(56 \cdot 0)+(58 \cdot 0) = 42$. (the filter doesn't work on the image but on a copy). As a graphical result, the initial pixel moved a pixel downwards.
![image-2.png](attachment:image-2.png)

These derivative filters are easy to implement using the standard convolution available in the scipy.ndimage.filters module. For example:

In [ ]:
from PIL import Image
from numpy import *
from scipy.ndimage import sobel
im = array(Image.open('roadmap.jpg').convert('L'))
# Sobel derivative filters
imx = zeros(im.shape)
sobel(im,1,imx)
imy = zeros(im.shape)
sobel(im,0,imy)
magnitude = sqrt(imx**2+imy**2)

This computes $x$ and $y$ derivatives and gradient magnitude using the Sobel filter. The second argument selects the $x$ or $y$ derivative, and the third stores the output. The figure below shows an image with derivatives computed using the Sobel filter as well as the gradient magnitude (last on the right). In the two derivative images, positive derivatives are shown with bright pixels and negative derivatives are dark. Gray areas have values close to zero.

In [ ]:
imgrey = Image.open('roadmap.jpg').convert('L')
plt.imshow(imgrey,cmap='gray')

In [ ]:
fig, axs = plt.subplots(1, 4,figsize=(14,14))
axs[0].imshow(Image.open('roadmap.jpg').convert('L'),cmap='gray')
axs[0].set_title('Original Image', size=16)
axs[1].imshow(imx,cmap='gray')
axs[1].set_title('x-derivative', size=16)
axs[2].imshow(imy,cmap='gray')
axs[2].set_title('y-derivative', size=16)
axs[3].imshow(magnitude,cmap='gray')
axs[3].set_title('magnitude', size=16)
plt.tight_layout()

Using this approach has the drawback that derivatives are taken on the scale determined by the image resolution. To be more robust to image noise and to compute derivatives at any scale, Gaussian derivative filters can be used:
$$I_x = I * G_{\sigma x} \ , \ \ I_y = I * G_{\sigma y} \ ,$$
where $G_{\sigma x}$ and $G_{\sigma y}$ are the $x$- and $y$-derivatives of the Gaussian kernel $G_{\sigma}(x,y)$ seen previously. (Work these derivatives out, as a small exercise).

These Gaussian filter functions we used for blurring earlier can also take extra arguments to compute Gaussian derivatives instead. To try this on an image, simply do:

In [ ]:
sigma = 10 # standard deviation
imx = zeros(im.shape)
gaussian_filter(im, (sigma,sigma), (0,1), imx)
imy = zeros(im.shape)
gaussian_filter(im, (sigma,sigma), (1,0), imy)
magnitude = sqrt(imx**2+imy**2)

Now we get the results:

In [ ]:
fig, axs = plt.subplots(1, 4,figsize=(14,14))
axs[0].imshow(Image.open('roadmap.jpg').convert('L'),cmap='gray')
axs[0].set_title('Original Image', size=16)
axs[1].imshow(imx,cmap='gray')
axs[1].set_title('x-derivative', size=16)
axs[2].imshow(imy,cmap='gray')
axs[2].set_title('y-derivative', size=16)
axs[3].imshow(magnitude,cmap='gray')
axs[3].set_title('magnitude', size=16)
plt.tight_layout()